# Experiment 8: contextual transformer targeted fine-tuning

This experiment tests whether targeted training is more effective when the contextual encoder itself can adapt. It uses the generic pretrained `distilbert-base-uncased` model, which is not fine-tuned on Financial PhraseBank. The experiment compares a normal fine-tuned transformer, a targeted fine-tuned transformer, and a matched random-control transformer. Hard training examples are selected from out-of-fold predictions; validation and test subsets are selected by uncertainty only. Validation macro-F1 is unweighted and is used for early stopping.

## Imports and reproducibility

In [1]:
import os
os.environ["HF_HOME"] = os.path.abspath("../.cache/huggingface")
os.environ["HF_HUB_CACHE"] = os.path.abspath("../.cache/huggingface/hub")
os.makedirs(os.environ["HF_HUB_CACHE"], exist_ok=True)

import copy
import pickle
import random
import sys

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import StratifiedKFold
from torch.optim import AdamW
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

sys.path.append("..")
from utils.hard_samples_identify import get_hard_sample_indices
from utils.preprocessing import LABELS, encode_labels

SEED = 42
MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 64
BATCH_SIZE = 16
BASELINE_EPOCHS = 3
TARGETED_EPOCHS = 2
LEARNING_RATE = 2e-5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
print("Device:", DEVICE)


Device: cpu


## Load data and tokenize

In [2]:
train_df = pd.read_csv("../data/train.csv")
val_df = pd.read_csv("../data/val.csv")
test_df = pd.read_csv("../data/test.csv")
y_train = encode_labels(train_df.label)
y_val = encode_labels(val_df.label)
y_test = encode_labels(test_df.label)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
def tokenize(frame):
    encoded = tokenizer(
        frame.sentence.tolist(), padding="max_length", truncation=True,
        max_length=MAX_LENGTH, return_tensors="pt",
    )
    return encoded["input_ids"], encoded["attention_mask"]

input_train, mask_train = tokenize(train_df)
input_val, mask_val = tokenize(val_df)
input_test, mask_test = tokenize(test_df)
print(input_train.shape, input_val.shape, input_test.shape)


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

torch.Size([2901, 64]) torch.Size([967, 64]) torch.Size([968, 64])


## Training and evaluation helpers

In [3]:
def loader(input_ids, attention_mask, labels, weights=None, shuffle=False):
    tensors = [input_ids, attention_mask, torch.tensor(labels, dtype=torch.long)]
    if weights is not None:
        tensors.append(torch.tensor(weights, dtype=torch.float32))
    return DataLoader(TensorDataset(*tensors), batch_size=BATCH_SIZE, shuffle=shuffle)

def new_model(seed=SEED):
    set_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=len(LABELS), id2label=dict(enumerate(LABELS)),
        label2id={label: i for i, label in enumerate(LABELS)},
    )
    return model.to(DEVICE)

@torch.no_grad()
def predict(model, input_ids, attention_mask, labels, return_loss=False):
    model.eval()
    predictions, probabilities, losses = [], [], []
    for batch in loader(input_ids, attention_mask, labels):
        ids, masks, batch_labels = [value.to(DEVICE) for value in batch]
        output = model(input_ids=ids, attention_mask=masks, labels=batch_labels)
        probabilities.append(torch.softmax(output.logits, dim=1).cpu().numpy())
        predictions.append(output.logits.argmax(dim=1).cpu().numpy())
        losses.append(output.loss.item())
    probs = np.concatenate(probabilities)
    preds = np.concatenate(predictions)
    return (probs, preds, float(np.mean(losses))) if return_loss else (probs, preds)

def macro_f1(model, input_ids, attention_mask, labels):
    _, predictions = predict(model, input_ids, attention_mask, labels)
    return f1_score(labels, predictions, average="macro")

def fit_model(model, input_ids, attention_mask, labels, val_data, sample_weights=None, epochs=3):
    train_loader = loader(input_ids, attention_mask, labels, sample_weights, shuffle=True)
    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
    steps = len(train_loader) * epochs
    schedule = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=max(1, steps // 10), num_training_steps=steps)
    best_state, best_score = copy.deepcopy(model.state_dict()), -np.inf
    for epoch in range(epochs):
        model.train()
        for batch in train_loader:
            ids, masks, batch_labels, *extra = [value.to(DEVICE) for value in batch]
            output = model(input_ids=ids, attention_mask=masks, labels=batch_labels)
            if extra:
                per_example_loss = torch.nn.functional.cross_entropy(output.logits, batch_labels, reduction="none")
                loss = (per_example_loss * extra[0]).mean()
            else:
                loss = output.loss
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); schedule.step()
        score = macro_f1(model, *val_data)
        print(f"epoch {epoch + 1}/{epochs} - val_macro_f1: {score:.4f}")
        if score > best_score:
            best_score = score
            best_state = copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    return model

def score(model, input_ids, attention_mask, labels, hard_indices):
    probabilities, predictions = predict(model, input_ids, attention_mask, labels)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "macro_f1": f1_score(labels, predictions, average="macro"),
        "hard_accuracy": accuracy_score(labels[hard_indices], predictions[hard_indices]),
        "hard_macro_f1": f1_score(labels[hard_indices], predictions[hard_indices], average="macro"),
    }, probabilities


## Fine-tune the baseline transformer

In [4]:
validation_data = (input_val, mask_val, y_val)
baseline_model = fit_model(new_model(SEED), input_train, mask_train, y_train, validation_data, epochs=BASELINE_EPOCHS)
_, baseline_train_probs = predict(baseline_model, input_train, mask_train, y_train)
baseline_val_probs, _ = predict(baseline_model, input_val, mask_val, y_val)
baseline_test_probs, _ = predict(baseline_model, input_test, mask_test, y_test)


W1027 12:14:05.150000 87976 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


epoch 1/3 - val_macro_f1: 0.7999
epoch 2/3 - val_macro_f1: 0.8216
epoch 3/3 - val_macro_f1: 0.8145


## Select hard examples without using training-set labels

In [5]:
# OOF models are short warm-up models used only for selection. The final targeted
# model is trained independently, so each training item is selected by a model
# that did not see it during its selection fold.
oof_probs = np.zeros((len(y_train), len(LABELS)))
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
for fold, (fit_idx, held_out_idx) in enumerate(skf.split(input_train, y_train)):
    fold_model = fit_model(
        new_model(SEED + fold + 1), input_train[fit_idx], mask_train[fit_idx], y_train[fit_idx],
        validation_data, epochs=1,
    )
    oof_probs[held_out_idx], _ = predict(
        fold_model, input_train[held_out_idx], mask_train[held_out_idx], y_train[held_out_idx]
    )
    del fold_model
    if DEVICE.type == "cuda": torch.cuda.empty_cache()

percentile = 20
train_hard_indices, train_cutoff = get_hard_sample_indices(oof_probs, percentile)
val_hard_indices, val_cutoff = get_hard_sample_indices(baseline_val_probs, percentile)
test_hard_indices, test_cutoff = get_hard_sample_indices(baseline_test_probs, percentile)
print(f"P{percentile}: train={len(train_hard_indices)} cutoff={train_cutoff:.4f}; "
      f"val={len(val_hard_indices)} cutoff={val_cutoff:.4f}; "
      f"test={len(test_hard_indices)} cutoff={test_cutoff:.4f}")


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


epoch 1/1 - val_macro_f1: 0.4463


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


epoch 1/1 - val_macro_f1: 0.4966


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


epoch 1/1 - val_macro_f1: 0.5622
P20: train=580 cutoff=0.1321; val=194 cutoff=0.5922; test=194 cutoff=0.5877


## Targeted and matched random-control fine-tuning

In [6]:
targeted_weights = np.ones(len(y_train), dtype=np.float32)
targeted_weights[train_hard_indices] = 1.5
random_indices = np.random.default_rng(SEED).choice(len(y_train), size=len(train_hard_indices), replace=False)
random_weights = np.ones(len(y_train), dtype=np.float32)
random_weights[random_indices] = 1.5

targeted_model = new_model(SEED + 10)
targeted_model.load_state_dict(copy.deepcopy(baseline_model.state_dict()))
targeted_model = fit_model(
    targeted_model, input_train, mask_train, y_train, validation_data,
    sample_weights=targeted_weights, epochs=TARGETED_EPOCHS,
)
random_model = new_model(SEED + 20)
random_model.load_state_dict(copy.deepcopy(baseline_model.state_dict()))
random_model = fit_model(
    random_model, input_train, mask_train, y_train, validation_data,
    sample_weights=random_weights, epochs=TARGETED_EPOCHS,
)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


epoch 1/2 - val_macro_f1: 0.8134
epoch 2/2 - val_macro_f1: 0.8232


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


epoch 1/2 - val_macro_f1: 0.8234
epoch 2/2 - val_macro_f1: 0.8176


## Compare baseline, targeted, and random-control models

In [7]:
rows = []
for strategy, model in [
    ("baseline", baseline_model),
    ("targeted", targeted_model),
    ("random_control", random_model),
]:
    overall, probabilities = score(model, input_test, mask_test, y_test, test_hard_indices)
    rows.append({
        "strategy": strategy, "percentile": percentile,
        "train_hard_count": len(train_hard_indices), "test_hard_count": len(test_hard_indices),
        "train_cutoff": train_cutoff, "test_cutoff": test_cutoff, **overall,
    })
results_df = pd.DataFrame(rows)
for column in ["accuracy", "macro_f1", "hard_accuracy", "hard_macro_f1"]:
    results_df[f"delta_{column}_vs_baseline"] = results_df[column] - results_df.loc[0, column]
print(results_df.to_string(index=False, float_format=lambda value: f"{value:.4f}"))
print("\nTargeted classification report")
_, targeted_predictions = predict(targeted_model, input_test, mask_test, y_test)
print(classification_report(y_test, targeted_predictions, target_names=LABELS))


      strategy  percentile  train_hard_count  test_hard_count  train_cutoff  test_cutoff  accuracy  macro_f1  hard_accuracy  hard_macro_f1  delta_accuracy_vs_baseline  delta_macro_f1_vs_baseline  delta_hard_accuracy_vs_baseline  delta_hard_macro_f1_vs_baseline
      baseline          20               580              194        0.1321       0.5877    0.8523    0.8332         0.5928         0.6014                      0.0000                      0.0000                           0.0000                           0.0000
      targeted          20               580              194        0.1321       0.5877    0.8533    0.8351         0.6031         0.6027                      0.0010                      0.0019                           0.0103                           0.0013
random_control          20               580              194        0.1321       0.5877    0.8523    0.8428         0.6031         0.6226                      0.0000                      0.0097                       

## Save results

In [8]:
os.makedirs("../models", exist_ok=True)
results_df.to_csv("../models/experiment_8_transformer_targeted_results.csv", index=False)
tokenizer.save_pretrained("../models/experiment_8_distilbert_tokenizer")
targeted_model.save_pretrained("../models/experiment_8_targeted_distilbert")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]